# Prédiction de la catégorie d'un film (horreur / comédie)

1. Audit et nettoyage
2. Découpage du jeu de données
3. Représentation du texte (TF-IDF)
4. Points de comparaison
5. Recherche des réglages avec KerasTuner
6. Recherche des réglages avec Optuna
7. Verdict sur le test
8. Sauvegarde

In [4]:
import json
import platform
import warnings
from pathlib import Path

import joblib
import keras_tuner as kt
import numpy as np
import optuna
import pandas as pd
import sklearn
import spacy
import tensorflow as tf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
import keras

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_PATH = Path("/home/projects/predict-film-category/data/textes_films_horreur_comedie.csv")
MODEL_DIR = Path("model")
SEED = 42
SEEDS = [0, 1, 2, 3, 4]
LABELS = {1: "horreur", 0: "comédie"}

try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except OSError:
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

## 1. Audit et nettoyage

In [5]:
df = pd.read_csv(DATA_PATH)

print("Dimensions :", df.shape)
print(df["cible"].map(LABELS).value_counts().to_string())
print()
print(df["resume"].str.len().describe().round(0).to_string())

Dimensions : (2400, 3)
cible
comédie    1200
horreur    1200

count    2400.0
mean      558.0
std       349.0
min       201.0
25%       335.0
50%       474.0
75%       679.0
max      7104.0


In [6]:
n_avant = len(df)

df = df.dropna(subset=["resume"])
df["resume"] = df["resume"].str.strip()
df = df[df["resume"] != ""]

n_doublons = int(df["resume"].duplicated().sum())
n_contradictoires = int((df.groupby("resume")["cible"].nunique() > 1).sum())
df = df.drop_duplicates(subset="resume").reset_index(drop=True)

print(f"{n_avant} lignes | {n_doublons} doublons supprimés | {n_contradictoires} résumé(s) à cibles contradictoires")
print(f"{len(df)} textes conservés")
print(df["cible"].map(LABELS).value_counts().to_string())

2400 lignes | 54 doublons supprimés | 0 résumé(s) à cibles contradictoires
2346 textes conservés
cible
comédie    1195
horreur    1151


In [7]:
def nettoyer_doc(doc):
    """Lemmes en minuscules, mots alphabétiques uniquement, sans mots vides."""
    return " ".join(t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop)


# Transformation sans apprentissage : pas de fuite entre apprentissage et test
df["propre"] = [nettoyer_doc(d) for d in nlp.pipe(df["resume"], batch_size=128)]
df = df[df["propre"].str.len() > 0].reset_index(drop=True)

print(df["propre"].iloc[0][:200])

roberto new job teacher nursery school child meet gianluigi mute roberto spirited bring tv class donkey take child unscheduled field trip father plant date isabella mother student soon pregnant moody 


## 2. Découpage du jeu de données

In [8]:
# 60 % apprentissage / 20 % test / 20 % production (mis de côté)
train, reste = train_test_split(df, test_size=0.40, stratify=df["cible"], random_state=SEED)
test, prod = train_test_split(reste, test_size=0.50, stratify=reste["cible"], random_state=SEED)

for nom, part in (("apprentissage", train), ("test", test), ("production", prod)):
    print(f"{nom:<14} {len(part):>5} textes | part d'horreur : {part['cible'].mean():.3f}")

y_train, y_test, y_prod = (p["cible"].to_numpy() for p in (train, test, prod))

apprentissage   1407 textes | part d'horreur : 0.490
test             469 textes | part d'horreur : 0.490
production       470 textes | part d'horreur : 0.491


## 3. Représentation du texte (TF-IDF)

In [9]:
vec = TfidfVectorizer()
X_train = vec.fit_transform(train["propre"])  # vocabulaire appris sur l'apprentissage seul
X_test = vec.transform(test["propre"])
X_prod = vec.transform(prod["propre"])

# Version dense pour les réseaux de neurones
X_train_d, X_test_d, X_prod_d = (m.toarray().astype("float32") for m in (X_train, X_test, X_prod))

print("Vocabulaire :", len(vec.vocabulary_), "mots")

Vocabulaire : 11807 mots


## 4. Points de comparaison

In [10]:
def marge_bruit(p, n):
    return 1.96 * np.sqrt(p * (1 - p) / n)


def indices_validation(graine):
    """Découpe l'apprentissage en deux : le test reste intact."""
    return train_test_split(np.arange(len(y_train)), test_size=0.20, stratify=y_train, random_state=graine)


def entrainer_reseau(modele, X, y):
    arret = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    modele.fit(X, y, epochs=40, batch_size=32, validation_split=0.10, callbacks=[arret], verbose=0)
    return modele


def predire_reseau(modele, X):
    return (modele.predict(X, verbose=0).ravel() > 0.5).astype(int)


def reseau_depart():
    modele = keras.Sequential([
        keras.Input(shape=(X_train_d.shape[1],)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    modele.compile(optimizer="adam", loss="binary_crossentropy")
    return modele


def scores_sklearn(fabrique):
    scores = []
    for graine in SEEDS:
        i_fit, i_val = indices_validation(graine)
        modele = fabrique().fit(X_train[i_fit], y_train[i_fit])
        scores.append(f1_score(y_train[i_val], modele.predict(X_train[i_val])))
    return scores


def scores_reseau(fabrique):
    scores = []
    for graine in SEEDS:
        keras.utils.set_random_seed(graine)
        i_fit, i_val = indices_validation(graine)
        modele = entrainer_reseau(fabrique(), X_train_d[i_fit], y_train[i_fit])
        scores.append(f1_score(y_train[i_val], predire_reseau(modele, X_train_d[i_val])))
    return scores


def tableau_bilan(resultats):
    n_val = len(indices_validation(0)[1])
    return pd.DataFrame([
        {"modèle": nom, "F1 moyen": np.mean(s), "écart-type": np.std(s), "marge de bruit": marge_bruit(np.mean(s), n_val)}
        for nom, s in resultats.items()
    ]).round(3)

In [11]:
resultats = {
    "Régression logistique": scores_sklearn(lambda: LogisticRegression(max_iter=2000)),
    "Bayes naïf": scores_sklearn(MultinomialNB),
    "Réseau de départ": scores_reseau(reseau_depart),
}
tableau_bilan(resultats)

W0000 00:00:1792677532.321146   55635 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


,modèle,F1 moyen,écart-type,marge de bruit
0,Régression logistique,0.842,0.018,0.043
1,Bayes naïf,0.859,0.023,0.041
2,Réseau de départ,0.840,0.019,0.043


## 5. Recherche des réglages avec KerasTuner

In [12]:
# Espace de recherche commun à KerasTuner et Optuna : (min, max[, pas])
ESPACE = {
    "n_layers": (1, 3),
    "units": (32, 256, 32),
    "dropout": (0.0, 0.5, 0.1),
    "l2": (1e-6, 1e-2),
    "lr": (1e-4, 1e-2),
}
N_ESSAIS = 20

i_fit, i_val = indices_validation(SEED)
X_fit, X_val = X_train_d[i_fit], X_train_d[i_val]
y_fit, y_val = y_train[i_fit], y_train[i_val]


def construire_reseau(n_layers, units, dropout, l2, lr):
    modele = keras.Sequential([keras.Input(shape=(X_train_d.shape[1],))])
    for _ in range(int(n_layers)):
        modele.add(keras.layers.Dense(int(units), activation="relu", kernel_regularizer=keras.regularizers.l2(l2)))
        modele.add(keras.layers.Dropout(dropout))
    modele.add(keras.layers.Dense(1, activation="sigmoid"))
    modele.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss="binary_crossentropy")
    return modele


def score_f1(modele):
    """Même protocole pour les deux outils : entraîner sur X_fit, mesurer le F1 sur X_val."""
    keras.utils.set_random_seed(SEED)
    entrainer_reseau(modele, X_fit, y_fit)
    return f1_score(y_val, predire_reseau(modele, X_val))


class RechercheF1(kt.HyperModel):
    def build(self, hp):
        e = ESPACE
        return construire_reseau(
            n_layers=hp.Int("n_layers", e["n_layers"][0], e["n_layers"][1]),
            units=hp.Int("units", e["units"][0], e["units"][1], step=e["units"][2]),
            dropout=hp.Float("dropout", e["dropout"][0], e["dropout"][1], step=e["dropout"][2]),
            l2=hp.Float("l2", e["l2"][0], e["l2"][1], sampling="log"),
            lr=hp.Float("lr", e["lr"][0], e["lr"][1], sampling="log"),
        )

    def fit(self, hp, modele, **kwargs):
        return {"val_f1": score_f1(modele)}

In [13]:
tuner = kt.RandomSearch(
    RechercheF1(),
    objective=kt.Objective("val_f1", "max"),
    max_trials=N_ESSAIS,
    seed=SEED,
    directory="kt_logs",
    project_name="film_categorie",
    overwrite=True,
)
tuner.search()

reglages_kt = tuner.get_best_hyperparameters(1)[0].values
score_kt = tuner.oracle.get_best_trials(1)[0].score
print(f"KerasTuner : F1 = {score_kt:.3f} | {reglages_kt}")

Trial 20 Complete [00h 00m 23s]
val_f1: 0.8405797101449275

Best val_f1 So Far: 0.8582089552238806
Total elapsed time: 00h 06m 51s
KerasTuner : F1 = 0.858 | {'n_layers': 2, 'units': 32, 'dropout': 0.2, 'l2': 1.2232117875737861e-05, 'lr': 0.005977728042983696}


## 6. Recherche des réglages avec Optuna

In [14]:
def objectif(essai):
    e = ESPACE
    reglages = {
        "n_layers": essai.suggest_int("n_layers", e["n_layers"][0], e["n_layers"][1]),
        "units": essai.suggest_int("units", e["units"][0], e["units"][1], step=e["units"][2]),
        "dropout": essai.suggest_float("dropout", e["dropout"][0], e["dropout"][1], step=e["dropout"][2]),
        "l2": essai.suggest_float("l2", e["l2"][0], e["l2"][1], log=True),
        "lr": essai.suggest_float("lr", e["lr"][0], e["lr"][1], log=True),
    }
    return score_f1(construire_reseau(**reglages))


etude = optuna.create_study(direction="maximize", sampler=optuna.samplers.RandomSampler(seed=SEED))
etude.optimize(objectif, n_trials=N_ESSAIS)

reglages_optuna = etude.best_params
score_optuna = etude.best_value
print(f"Optuna : F1 = {score_optuna:.3f} | {reglages_optuna}")

Optuna : F1 = 0.855 | {'n_layers': 3, 'units': 224, 'dropout': 0.0, 'l2': 2.7155819552829413e-05, 'lr': 0.0001705053926026929}


In [15]:
marge = marge_bruit(max(score_kt, score_optuna), len(y_val))
ecart = abs(score_kt - score_optuna)

print(f"KerasTuner {score_kt:.3f} | Optuna {score_optuna:.3f} | écart {ecart:.3f} | marge de bruit ±{marge:.3f}")
print("Écart inférieur à la marge de bruit : outils équivalents." if ecart < marge
      else "Écart supérieur à la marge de bruit : un outil est meilleur.")

meilleurs_reglages = reglages_kt if score_kt >= score_optuna else reglages_optuna

KerasTuner 0.858 | Optuna 0.855 | écart 0.003 | marge de bruit ±0.041
Écart inférieur à la marge de bruit : outils équivalents.


## 7. Verdict sur le test

In [16]:
# Comparaison sur 5 graines (même protocole qu'à l'étape 4)
resultats["Réseau optimisé"] = scores_reseau(lambda: construire_reseau(**meilleurs_reglages))
bilan = tableau_bilan(resultats)
bilan

,modèle,F1 moyen,écart-type,marge de bruit
0,Régression logistique,0.842,0.018,0.043
1,Bayes naïf,0.859,0.023,0.041
2,Réseau de départ,0.840,0.019,0.043
3,Réseau optimisé,0.846,0.020,0.042


In [18]:
moyennes = {nom: np.mean(s) for nom, s in resultats.items()}
meilleur_simple = max(
    ("Régression logistique", "Bayes naïf"),
    key=lambda nom: moyennes[nom],
)
n_val = len(indices_validation(0)[1])

for reference in (meilleur_simple, "Réseau de départ"):
    ecart = moyennes["Réseau optimisé"] - moyennes[reference]
    marge = marge_bruit(moyennes["Réseau optimisé"], n_val)
    verdict = "meilleur" if ecart > marge else "écart inférieur à la marge de bruit" if abs(ecart) <= marge else "moins bon"
    print(f"Réseau optimisé vs {reference} : écart {ecart:+.3f} (marge ±{marge:.3f}) -> {verdict}")

Réseau optimisé vs Bayes naïf : écart -0.012 (marge ±0.042) -> écart inférieur à la marge de bruit
Réseau optimisé vs Réseau de départ : écart +0.006 (marge ±0.042) -> écart inférieur à la marge de bruit


In [19]:
# Entraînement final sur tout l'apprentissage, une seule mesure sur le test
keras.utils.set_random_seed(SEED)
modele_final = entrainer_reseau(construire_reseau(**meilleurs_reglages), X_train_d, y_train)

pred_test = predire_reseau(modele_final, X_test_d)
f1_test = f1_score(y_test, pred_test)
marge_test = marge_bruit(f1_test, len(y_test))
print(f"F1 sur le test : {f1_test:.3f} ± {marge_test:.3f} (n = {len(y_test)})")

F1 sur le test : 0.848 ± 0.032 (n = 469)


## 8. Sauvegarde

In [20]:
MODEL_DIR.mkdir(exist_ok=True)

modele_final.save(MODEL_DIR / "model.keras")
joblib.dump(vec, MODEL_DIR / "tfidf.joblib")

metrics = {
    "model_version": "1.0.0",
    "f1_test": round(float(f1_test), 4),
    "test_size": int(len(y_test)),
    "noise_margin": round(float(marge_test), 4),
    "best_params": meilleurs_reglages,
    "library_versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__,
        "tensorflow": tf.__version__,
        "spacy": spacy.__version__,
        "joblib": joblib.__version__,
    },
}
with open(MODEL_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False, default=lambda o: o.item())

train[["resume", "cible"]].sample(50, random_state=SEED).to_csv(
    MODEL_DIR / "reference_sample.csv", index=False, encoding="utf-8"
)

print(sorted(p.name for p in MODEL_DIR.iterdir()))

['metrics.json', 'model.keras', 'reference_sample.csv', 'tfidf.joblib']
